# LangChain Q&A Agent with Gemini
Conversational Q&A using Google Gemini + LangChain with memory.

In [ ]:
%pip install -q langchain langchain-google-genai langchain-community langchain-core python-dotenv

In [ ]:
import sys
print("Python:", sys.executable)
print("Version:", sys.version)

import subprocess
result = subprocess.run([sys.executable, "-m", "pip", "show", "langchain", "langchain-core", "langchain-google-genai"], capture_output=True, text=True)
print(result.stdout)

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0.3,
)

print("✅ Gemini LLM initialized.")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-flash",
    google_api_key=GOOGLE_API_KEY,
    temperature=0.3,
)

print("✅ Gemini LLM initialized.")

## Simple Q&A

In [ ]:
from langchain_core.messages import HumanMessage

question = "What is LangChain and why is it useful?"
response = llm.invoke([HumanMessage(content=question)])

print(f"Q: {question}")
print(f"A: {response.content}")

## Conversational Q&A with Memory

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer questions clearly and concisely."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

chain = prompt | llm
store = {}

def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

conversation = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

def chat(question, session_id="default"):
    response = conversation.invoke(
        {"input": question},
        config={"configurable": {"session_id": session_id}},
    )
    print(f"You: {question}")
    print(f"Gemini: {response.content}")
    print("-" * 60)

print("✅ Conversational chain ready.")

In [ ]:
chat("What is Retrieval-Augmented Generation?")

In [ ]:
chat("Can you give me a real-world example of that?")

In [ ]:
chat("What are the main limitations?")

## Q&A over Custom Text (Mini-RAG)

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

custom_text = """
LangChain is an open-source framework for building applications powered by large language models (LLMs).
It provides tools for chaining LLM calls, managing memory, connecting to external data sources, and building agents.
LangChain supports many LLM providers including OpenAI, Google Gemini, Anthropic Claude, and open-source models.
Key components include: Chains, Agents, Memory, Retrievers, and Prompts.
LangChain Expression Language (LCEL) is the modern way to compose chains using a pipe-like syntax.
"""

rag_prompt = PromptTemplate.from_template(
    "Use the context below to answer the question.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"
)

rag_chain = (
    {"context": RunnablePassthrough(), "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

def ask_docs(question):
    response = rag_chain.invoke({"context": custom_text, "question": question})
    print(f"Q: {question}")
    print(f"A: {response}")
    print("-" * 60)

ask_docs("What are the key components of LangChain?")

In [ ]:
ask_docs("Which LLM providers does LangChain support?")